<a href="https://colab.research.google.com/github/raymingchen/Finding-Partial-and-Global-Optimal-Combinations-for-Generalized-Non-parametric-Regressions/blob/main/comparison1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

# ---------------------------------------------------------
# 1. DATA (CO → DR)
# ---------------------------------------------------------
x_raw = np.array([
    100, 96.82, 93.64, 90.04, 86.61, 83.51, 79.57, 77.3, 75.63, 72.09,
    70.95, 68.21, 61.5, 59.79, 58.08, 56.37, 54.48, 52.58, 47.15, 42.33,
    43.04, 42.68, 40.8, 38.91, 37.03, 35.51
], dtype=float)

y = np.array([
    23.1, 28.11, 25.27, 20.33, 23.08, 27.68, 24.82, 29.17, 30.81, 29.09,
    29.27, 25.17, 22.98, 22.62, 25.89, 25.8, 20.8, 27.59, 31.46, 25.81,
    21.05, 24.35, 30.99, 23.54, 21.1, 28.86
], dtype=float)

# Normalize x to [0,1]
x = (x_raw - x_raw.min()) / (x_raw.max() - x_raw.min())

# Sort
idx = np.argsort(x)
x = x[idx]
y = y[idx]

# ---------------------------------------------------------
# 2. KERNELS
# ---------------------------------------------------------
def triangular(u):
    return np.maximum(1 - np.abs(u), 0)

def epanechnikov(u):
    return np.maximum(0.75 * (1 - u**2), 0)

def biweight(u):
    return np.maximum((15/16) * (1 - u**2)**2, 0)

kernels = {
    "Triangular": triangular,
    "Epanechnikov": epanechnikov,
    "Biweight": biweight
}

# ---------------------------------------------------------
# 3. CASE STRUCTURES
# ---------------------------------------------------------
def T_case1(xi, xj):
    return np.abs(xi - xj)

def T_case3(xi, xj):
    return np.abs(xi - xj)

# ---------------------------------------------------------
# 4. GENERALIZED ESTIMATOR
# ---------------------------------------------------------
def generalized_estimator(x0, x, y, kernel, alpha, case):
    w = []
    for xi in x:
        d = np.abs(x0 - xi)
        u = d / alpha
        w.append(kernel(u))

    w = np.array(w)
    if w.sum() == 0:
        return np.mean(y)
    return np.sum(w * y) / np.sum(w)

def generalized_predict(x, y, kernel, alpha, case):
    return np.array([generalized_estimator(x0, x, y, kernel, alpha, case) for x0 in x])

# ---------------------------------------------------------
# 5. AUTO-SELECT α USING CV (returns α only)
# ---------------------------------------------------------
def cv_mse_for_alpha(x, y, kernel, case, alpha):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mse_list = []

    for train_idx, test_idx in kf.split(x):
        x_train, y_train = x[train_idx], y[train_idx]
        x_test, y_test = x[test_idx], y[test_idx]

        y_pred = [generalized_estimator(x0, x_train, y_train, kernel, alpha, case)
                  for x0 in x_test]

        mse_list.append(mean_squared_error(y_test, y_pred))

    return np.mean(mse_list)

def auto_select_alpha(x, y, kernel, case):
    alpha_grid = np.linspace(0.01, 0.5, 60)
    best_alpha = None
    best_mse = float("inf")

    for alpha in alpha_grid:
        mse = cv_mse_for_alpha(x, y, kernel, case, alpha)
        if mse < best_mse:
            best_mse = mse
            best_alpha = alpha

    return best_alpha

# ---------------------------------------------------------
# 6. CLASSICAL KERNEL ESTIMATORS
# ---------------------------------------------------------
def nadaraya_watson(x, y, h, kernel):
    yhat = []
    for x0 in x:
        u = (x0 - x) / h
        w = kernel(u)
        if w.sum() == 0:
            yhat.append(np.mean(y))
        else:
            yhat.append(np.sum(w * y) / np.sum(w))
    return np.array(yhat)

def gasser_mueller(x, y, h, kernel):
    return np.array([
        np.sum(kernel((x0 - x) / h) * y) / np.sum(kernel((x0 - x) / h))
        for x0 in x
    ])

def priestley_chao(x, y, h, kernel):
    return np.array([
        np.sum(kernel((x0 - x) / h) * y) / np.sum(kernel((x0 - x) / h))
        for x0 in x
    ])

# ---------------------------------------------------------
# 7. MACHINE LEARNING MODELS
# ---------------------------------------------------------
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def ml_models(x, y):
    X = x.reshape(-1, 1)
    results = {}

    def metrics(y, yhat):
        mse = mean_squared_error(y, yhat)
        return (
            r2_score(y, yhat),
            mse,
            np.sqrt(mse),
            mean_absolute_error(y, yhat)
        )

    lr = LinearRegression().fit(X, y)
    results["Linear"] = metrics(y, lr.predict(X))

    poly = PolynomialFeatures(3)
    Xp = poly.fit_transform(X)
    lr2 = LinearRegression().fit(Xp, y)
    results["Poly(3)"] = metrics(y, lr2.predict(Xp))

    svr = SVR(kernel='rbf', C=10, gamma=0.1).fit(X, y)
    results["SVR(RBF)"] = metrics(y, svr.predict(X))

    rf = RandomForestRegressor(n_estimators=300).fit(X, y)
    results["RandomForest"] = metrics(y, rf.predict(X))

    xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3)
    xgb.fit(X, y)
    results["XGBoost"] = metrics(y, xgb.predict(X))

    return results

# ---------------------------------------------------------
# 8. RUN EVERYTHING
# ---------------------------------------------------------
results = {}

# Generalized estimators
for case in [1, 3]:
    for kname, kernel in kernels.items():
        alpha_opt = auto_select_alpha(x, y, kernel, case)
        yhat = generalized_predict(x, y, kernel, alpha_opt, case)
        mse = mean_squared_error(y, yhat)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y, yhat)
        r2 = r2_score(y, yhat)
        key = f"CASE {case} · {kname} · α={alpha_opt:.3f}"
        results[key] = (r2, mse, rmse, mae)

# Classical estimators
h = 0.10
for name, func in {
    "Nadaraya-Watson": nadaraya_watson,
    "Gasser-Mueller": gasser_mueller,
    "Priestley-Chao": priestley_chao
}.items():
    yhat = func(x, y, h, epanechnikov)
    mse = mean_squared_error(y, yhat)
    results[name] = (
        r2_score(y, yhat), mse, np.sqrt(mse), mean_absolute_error(y, yhat)
    )

# ML models
results.update(ml_models(x, y))

# ---------------------------------------------------------
# 9. PRINT TABLE
# ---------------------------------------------------------
print("\n=== FULL COMPARISON TABLE (R², MSE, RMSE, MAE) ===")
for method, (r2, mse, rmse, mae) in results.items():
    print(f"{method:40s}  R²={r2:.4f}  MSE={mse:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")



=== FULL COMPARISON TABLE (R², MSE, RMSE, MAE) ===
CASE 1 · Triangular · α=0.018             R²=0.9778  MSE=0.2393  RMSE=0.4892  MAE=0.1513
CASE 1 · Epanechnikov · α=0.018           R²=0.9683  MSE=0.3422  RMSE=0.5850  MAE=0.1806
CASE 1 · Biweight · α=0.384               R²=0.0377  MSE=10.3913  RMSE=3.2236  MAE=2.7508
CASE 3 · Triangular · α=0.018             R²=0.9778  MSE=0.2393  RMSE=0.4892  MAE=0.1513
CASE 3 · Epanechnikov · α=0.018           R²=0.9683  MSE=0.3422  RMSE=0.5850  MAE=0.1806
CASE 3 · Biweight · α=0.384               R²=0.0377  MSE=10.3913  RMSE=3.2236  MAE=2.7508
Nadaraya-Watson                           R²=0.3484  MSE=7.0366  RMSE=2.6527  MAE=2.2006
Gasser-Mueller                            R²=0.3484  MSE=7.0366  RMSE=2.6527  MAE=2.2006
Priestley-Chao                            R²=0.3484  MSE=7.0366  RMSE=2.6527  MAE=2.2006
Linear                                    R²=0.0000  MSE=10.7977  RMSE=3.2860  MAE=2.7797
Poly(3)                                   R²=0.0504  MS